In [1]:
!pip install -q transformers datasets sentence-transformers accelerate torch evaluate

In [2]:
import os
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Check if files are already uploaded
needed_files = ["train.csv", "validation.csv", "test.csv"]
missing = [f for f in needed_files if not os.path.exists(os.path.join(DATA_DIR, f)) and not os.path.exists(f"/content/{f}")]

if missing:
    print(f"Please upload your missing files: {missing}")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(f"/content/{filename}", os.path.join(DATA_DIR, filename))
else:
    # If they are in /content/, move them to /content/data/
    for f in needed_files:
        if os.path.exists(f"/content/{f}") and not os.path.exists(os.path.join(DATA_DIR, f)):
            os.rename(f"/content/{f}", os.path.join(DATA_DIR, f))

print("\nFiles in /content/data/:", os.listdir(DATA_DIR))


Files in /content/data/: ['train.csv', 'test.csv', 'validation.csv']


In [3]:
import os
import pandas as pd
from datasets import Dataset
DATA_DIR = "/content/data"
INSTRUCTION = "Identify the type of this legal contract clause in a few words.\n\nClause: "
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
def format_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["input_text"] = INSTRUCTION + df["clause_text"].astype(str)
    # Format labels to clean readable text
    df["target_text"] = (
        df["label"]
        .astype(str)
        .str.replace("__", " - ")
        .str.replace("_", " ")
        .str.strip()
        .str.lower()
    )
    return df
train_df = format_targets(train_df)
val_df = format_targets(val_df)
test_df = format_targets(test_df)

print(f"Loaded Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Unique categories: {train_df['target_text'].nunique()}")

Loaded Train: 6209 | Val: 782 | Test: 708
Unique categories: 495


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 32
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
# CRITICAL FIX: use_fast=False eliminates the Python 3.13 Rust overflow bug
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["input_text", "target_text"]], preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["input_text", "target_text"])
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


Device: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/6209 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/708 [00:00<?, ? examples/s]

In [5]:
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

MODEL_DIR = "/content/models/pactguard-flant5"

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,           # Prevents NaN loss
    predict_with_generate=False, # Evaluates validation loss fast in seconds
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    bf16=use_bf16,
    fp16=False,                  # Strictly False for T5
    report_to="none",
    save_total_limit=1
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)
print("Trainer successfully configured!")

Trainer successfully configured!


In [6]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.615048,0.663125
2,0.742303,0.463621
3,0.472232,0.395368
4,0.344870,0.377702


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1556, training_loss=1.238885242712222, metrics={'train_runtime': 2403.6318, 'train_samples_per_second': 10.333, 'train_steps_per_second': 0.647, 'total_flos': 7321821487718400.0, 'train_loss': 1.238885242712222, 'epoch': 4.0})

In [7]:
print("Evaluating on Test Set...")
test_results = trainer.evaluate(test_dataset)
print("\n--- Test Results ---")
for k, v in test_results.items():
    print(f"{k}: {v}")
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"\nModel saved to: {MODEL_DIR}")

Evaluating on Test Set...


Training Loss,Validation Loss,Epoch
0.344870,0.308948,4



--- Test Results ---
eval_loss: 0.3089475631713867


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: /content/models/pactguard-flant5


In [8]:
#Inference Test
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def test_clause(clause_text):
    inputs = tokenizer(INSTRUCTION + clause_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_new_tokens=32)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print("Result 1:", test_clause("The tenant may not sublease the premises without prior written consent of the landlord."))

print("Result 2:", test_clause("Tuition fees are non-refundable after the second week of the semester."))

print("Result 3:", test_clause("Company shall defend, indemnify, and hold harmless Customer against any patent infringement claims."))

Result 1: anti assignment - sublicense bli granted
Result 2: insurance - fee waived deemed
Result 3: liability cap - patent intellectual patent applications
